# Data Pre-processing for Transformer-Based Sentiment Classification

**Important NOTE:** This notebook prepares the dataset for a transformer model.
Since transformers work directly with **contextual text**, the preprocessing is kept **minimal**.

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

/Users/fatmaoztel/Desktop/Fatima Projects/cs-conversation-sentiment-analysis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the data

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (970, 11)
Test shape: (30, 11)


First load the train and test sets so that the preprocessing pipeline is applied consistently.

## 2. Select target and modeling columns

In [3]:
target_col = "customer_sentiment"
text_col = "conversation"

metadata_cols = [
    "issue_area",
    "issue_category",
    "issue_complexity",
    "agent_experience_level",
    "product_category"
]

selected_columns = [text_col] + metadata_cols + [target_col]

train_model_df = train_df[selected_columns].copy()
test_model_df = test_df[selected_columns].copy()

print("Train modeling shape:", train_model_df.shape)
print("Test modeling shape:", test_model_df.shape)
display(train_model_df.head())

Train modeling shape: (970, 7)
Test modeling shape: (30, 7)


,conversation,issue_area,issue_category,issue_complexity,agent_experience_level,product_category,customer_sentiment
0,"Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?\n\nCustomer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG),...",Login and Account,Mobile Number and Email Verification,medium,junior,Appliances,neutral
1,Agent: Thank you for calling BrownBox customer support. My name is Alex. How may I assist you today?\n\nCustomer: Hi Alex. I recently received an email from BrownBox requesting me to ship back the...,Cancellations and returns,Pickup and Shipping,less,junior,Electronics,neutral
2,"Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How may I assist you today?\n\nCustomer: Hi Sarah, I am calling because I am unable to click the 'Cancel' button for my Ju...",Cancellations and returns,Replacement and Return Process,medium,experienced,Appliances,neutral
3,"Customer: Hi, I am facing an issue while logging into my account. I am getting an error message saying that I have exceeded the number of attempts to enter the correct verification code.\n\nAgent:...",Login and Account,Login Issues and Error Messages,less,inexperienced,Appliances,neutral
4,"Agent: Thank you for contacting BrownBox customer support. My name is Sarah. How can I assist you today?\n\nCustomer: Hi Sarah, I have an issue with my order. I received a BP monitor, but the deli...",Order,Order Delivery Issues,medium,experienced,Electronics,negative


In this version, the conversation remains the main text input, but selected metadata columns are converted into text and added as a prefix. This keeps the transformer architecture unchanged while giving the model useful issue-related context.

## Check missing values

In [4]:
train_missing = train_model_df.isnull().sum().sort_values(ascending=False)
test_missing = test_model_df.isnull().sum().sort_values(ascending=False)

print("Train missing values:")
display(train_missing[train_missing > 0])

print("Test missing values:")
display(test_missing[test_missing > 0])

Train missing values:


Series([], dtype: int64)

Test missing values:


Series([], dtype: int64)

Missing values are checked before tokenization because null text values would cause errors in later steps. No missing values are found, so no imputation is needed.

## Handle duplicate rows

In [5]:
print("Duplicate rows in train_model_df:", train_model_df.duplicated().sum())
print("Duplicate conversation texts in train_model_df:", train_model_df[text_col].duplicated().sum())

Duplicate rows in train_model_df: 0
Duplicate conversation texts in train_model_df: 2


In [6]:
train_model_df = train_model_df.drop_duplicates().reset_index(drop=True)

print("Train shape after removing duplicate rows:", train_model_df.shape)

Train shape after removing duplicate rows: (970, 7)


Exact duplicate rows were removed. It was detected on the EDA nb as well.

## Encode sentiment labels

In [7]:
label_mapping = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_model_df["sentiment_label"] = train_model_df[target_col].map(label_mapping)
test_model_df["sentiment_label"] = test_model_df[target_col].map(label_mapping)

print("Label mapping:", label_mapping)
display(train_model_df[[target_col, "sentiment_label"]].drop_duplicates().sort_values("sentiment_label"))

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}


,customer_sentiment,sentiment_label
4,negative,0
0,neutral,1
143,positive,2


Labels are converted to numeric values because transformer classifiers require numerical targets during training.

## Create a validation set

In [8]:
train_split_df, val_split_df = train_test_split(
    train_model_df,
    test_size=0.2,
    random_state=42,
    stratify=train_model_df["sentiment_label"]
)

train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

print("Train split shape:", train_split_df.shape)
print("Validation split shape:", val_split_df.shape)

Train split shape: (776, 8)
Validation split shape: (194, 8)


In [9]:
print("Training set sentiment distribution:")
(train_split_df[target_col].value_counts(normalize=True) * 100).round(2)

Training set sentiment distribution:


customer_sentiment
neutral     55.93
negative    42.40
positive     1.68
Name: proportion, dtype: float64

In [10]:
print("Validation set sentiment distribution:")
(val_split_df[target_col].value_counts(normalize=True) * 100).round(2)

Validation set sentiment distribution:


customer_sentiment
neutral     55.67
negative    42.27
positive     2.06
Name: proportion, dtype: float64

80-20 split has been chosen. 80-20 split is used to keep most samples for training while reserving a meaningful validation set. Stratified splitting is chosen because the dataset is imbalanced, especially for the positive class.

## Build metadata prefix

In [11]:
def build_metadata_prefix(row):
    return (
        f"Issue area: {row['issue_area']}\n"
        f"Issue category: {row['issue_category']}\n"
        f"Issue complexity: {row['issue_complexity']}\n"
        f"Agent experience: {row['agent_experience_level']}\n"
        f"Product category: {row['product_category']}\n\n"
        f"Conversation: {row['conversation']}"
    )

In [12]:
train_split_df["conversation_with_metadata"] = train_split_df.apply(build_metadata_prefix, axis=1)
val_split_df["conversation_with_metadata"] = val_split_df.apply(build_metadata_prefix, axis=1)
test_model_df["conversation_with_metadata"] = test_model_df.apply(build_metadata_prefix, axis=1)

display(train_split_df[["issue_area", "issue_category", "conversation_with_metadata"]].head(2))

,issue_area,issue_category,conversation_with_metadata
0,Cancellations and returns,Pickup and Shipping,"Issue area: Cancellations and returns\nIssue category: Pickup and Shipping\nIssue complexity: medium\nAgent experience: junior\nProduct category: Appliances\n\nConversation: Customer: Hi, I'm call..."
1,Cancellations and returns,Pickup and Shipping,"Issue area: Cancellations and returns\nIssue category: Pickup and Shipping\nIssue complexity: less\nAgent experience: junior\nProduct category: Appliances\n\nConversation: Customer: Hi, I received..."


The selected metadata columns are converted into short text descriptions and added before the conversation. This gives the model structured context without changing the transformer architecture.

In [14]:
sample_idx = 0

print("FULL CONVERSATION:\n")
print(train_split_df.loc[sample_idx, text_col])

print("\n" + "="*100 + "\n")

print("WITH METADATA:\n")
print(train_split_df.loc[sample_idx, "conversation_with_metadata"])

FULL CONVERSATION:

Customer: Hi, I'm calling because I have an issue with my recent order. I purchased an air cooler from BrownBox, but I haven't received the refund in my bank account yet.

Agent: Hi, I'm sorry to hear that. I can definitely help you with that. May I please have your order number so I can look into this issue for you?

Customer: Sure, it's BB123456789.

Agent: Thank you for that. I'm sorry to hear that you haven't received your refund yet. Could you please confirm the date when you returned the air cooler?

Customer: I returned it on August 1st, 2021.

Agent: Thank you for that information. I can see that the refund was processed on August 5th, 2021, and it should have been credited to your bank account within 5-7 business days. Can you please confirm the bank account details you provided for the refund?

Customer: Yes, the bank account details I provided were correct.

Agent: I apologize for the inconvenience caused. Sometimes, it may take a little longer for the re

## Light text normalization

In [15]:
def normalize_text_for_transformer(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

train_split_df["conversation_clean"] = train_split_df["conversation_with_metadata"].apply(normalize_text_for_transformer)
val_split_df["conversation_clean"] = val_split_df["conversation_with_metadata"].apply(normalize_text_for_transformer)
test_model_df["conversation_clean"] = test_model_df["conversation_with_metadata"].apply(normalize_text_for_transformer)

display(train_split_df[["conversation_with_metadata", "conversation_clean"]].head(2))

,conversation_with_metadata,conversation_clean
0,"Issue area: Cancellations and returns\nIssue category: Pickup and Shipping\nIssue complexity: medium\nAgent experience: junior\nProduct category: Appliances\n\nConversation: Customer: Hi, I'm call...","issue area: cancellations and returns issue category: pickup and shipping issue complexity: medium agent experience: junior product category: appliances conversation: customer: hi, i'm calling bec..."
1,"Issue area: Cancellations and returns\nIssue category: Pickup and Shipping\nIssue complexity: less\nAgent experience: junior\nProduct category: Appliances\n\nConversation: Customer: Hi, I received...","issue area: cancellations and returns issue category: pickup and shipping issue complexity: less agent experience: junior product category: appliances conversation: customer: hi, i received an ema..."


After adding the metadata prefix, only light normalization is applied. We keep the original wording and sequence structure because transformers learn context from the text itself. We do not remove stopwords, stem, or lemmatize.

## Load the pretrained transformer tokenizer

In [16]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Tokenizer loaded from:", model_checkpoint)

Tokenizer loaded from: distilbert-base-uncased


A pretrained tokenizer is used because transformer models already come with their own vocabulary and tokenization rules. This is more appropriate than creating a custom tokenizer. --> No need to create from scratch.

The tokenizer for distilbert-base-uncased converts raw text into the numerical format DistilBERT can understand.

What it does:

* lowercases the text: Because the model is uncased, it treats uppercase and lowercase the same.
* splits text into tokens: It breaks the sentence into smaller units, often words or subwords.
* uses subword tokenization: If a word is rare or unknown, it can split it into smaller pieces instead of failing completely.
* maps tokens to IDs: Each token is converted into an integer from the model vocabulary.
* adds special tokens: It adds tokens such as [CLS] and [SEP] if needed by the model format.
* creates attention masks: It marks which parts are real text and which parts are padding.
* pads and truncates sequences: it makes all inputs the same length for batch processing.

## Check token length

In [17]:
token_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in train_split_df["conversation_clean"]]

print("Mean token length:", round(np.mean(token_lengths), 2))
print("Median token length:", round(np.median(token_lengths), 2))
print("95th percentile:", int(np.percentile(token_lengths, 95)))
print("Max token length:", np.max(token_lengths))

Token indices sequence length is longer than the specified maximum sequence length for this model (791 > 512). Running this sequence through the model will result in indexing errors


Mean token length: 522.6
Median token length: 508.0
95th percentile: 744
Max token length: 1345


In [18]:
max_length = min(512, int(np.percentile(token_lengths, 95)))
print("Chosen max_length:", max_length)

Chosen max_length: 512


The maximum sequence length is chosen from the training distribution and capped at 512 tokens due to model restriction.

We did that to balance two things:
- keeping enough of each conversation
- avoiding unnecessary computation

Why not use the full maximum length?
- some conversations are much longer than most others
- if we pad everything to the longest conversation, training becomes slower, memory usage increases a lot
- most rows would contain too much padding
- the 95th percentile keeps most samples almost complete
BUT in this case, due to model restriction, we will use 512 token

Why use the training distribution? (IMPORTANT FOR NO DATA LEAKAGE)
- it shows what a typical conversation length looks like by choosing from the training set avoids using information from validation/test unnecessarily

Should we cap at less than 512?
- transformer cost grows with sequence length: longer sequences require much more memory and time
- if memory becomes a problem, we can try 384 --> LETS SEE

## Tokenize the text

In [19]:
train_encodings = tokenizer(
    train_split_df["conversation_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="np"
)

val_encodings = tokenizer(
    val_split_df["conversation_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="np"
)

test_encodings = tokenizer(
    test_model_df["conversation_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="np"
)

print("Train input_ids shape:", train_encodings["input_ids"].shape)
print("Validation input_ids shape:", val_encodings["input_ids"].shape)
print("Test input_ids shape:", test_encodings["input_ids"].shape)

Train input_ids shape: (776, 512)
Validation input_ids shape: (194, 512)
Test input_ids shape: (30, 512)


input_ids and attention_mask are the main inputs a transformer needs.

**input_ids:**
- these are the token numbers produced by the tokenizer
- each word or subword is converted into an integer ID from the model vocabulary
- this is how the model reads the text

**attention_mask:**
- this tells the model which positions are real text and which are just padding
- 1 means real token
- 0 means padded position

## Prepare final model inputs

In [20]:
X_train_input_ids_metadata_prefix = train_encodings["input_ids"]
X_train_attention_mask_metadata_prefix = train_encodings["attention_mask"]

X_val_input_ids_metadata_prefix = val_encodings["input_ids"]
X_val_attention_mask_metadata_prefix = val_encodings["attention_mask"]

X_test_input_ids_metadata_prefix = test_encodings["input_ids"]
X_test_attention_mask_metadata_prefix = test_encodings["attention_mask"]

y_train_metadata_prefix = train_split_df["sentiment_label"].values
y_val_metadata_prefix = val_split_df["sentiment_label"].values
y_test_metadata_prefix = test_model_df["sentiment_label"].values

print("X_train_input_ids shape:", X_train_input_ids_metadata_prefix.shape)
print("X_train_attention_mask shape:", X_train_attention_mask_metadata_prefix.shape)
print("X_val_input_ids shape:", X_val_input_ids_metadata_prefix.shape)
print("X_test_input_ids shape:", X_test_input_ids_metadata_prefix.shape)
print("y_train shape:", y_train_metadata_prefix.shape)
print("y_val shape:", y_val_metadata_prefix.shape)
print("y_test shape:", y_test_metadata_prefix.shape)

X_train_input_ids shape: (776, 512)
X_train_attention_mask shape: (776, 512)
X_val_input_ids shape: (194, 512)
X_test_input_ids shape: (30, 512)
y_train shape: (776,)
y_val shape: (194,)
y_test shape: (30,)


These arrays are the final inputs for transformer training.

## SUMMARY

- **"conversation"** was selected as the main model input because sentiment is expressed in the dialogue.
- Missing values and duplicate rows were checked to improve data quality.
- Sentiment labels were encoded numerically for classification.
- A stratified validation split was created because the dataset is imbalanced.
- Only light text normalization was applied, since transformers should receive text close to its original form.
- A pretrained DistilBERT tokenizer was used instead of manual vocabulary building.

## Save processed arrays

The final tokenized arrays are saved so the model-training notebook can load them directly without repeating preprocessing.

In [21]:
np.save("X_train_input_ids.npy", X_train_input_ids_metadata_prefix)
np.save("X_train_attention_mask.npy", X_train_attention_mask_metadata_prefix)
np.save("X_val_input_ids.npy", X_val_input_ids_metadata_prefix)
np.save("X_val_attention_mask.npy", X_val_attention_mask_metadata_prefix)
np.save("X_test_input_ids.npy", X_test_input_ids_metadata_prefix)
np.save("X_test_attention_mask.npy", X_test_attention_mask_metadata_prefix)

np.save("y_train.npy", y_train_metadata_prefix)
np.save("y_val.npy", y_val_metadata_prefix)
np.save("y_test.npy", y_test_metadata_prefix)

print("Processed transformer arrays were saved successfully.")

Processed transformer arrays were saved successfully.
